<a href="https://colab.research.google.com/github/twyeh/highenergy/blob/main/QCDF_Branching_Ratios_and_Asymmetries.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

This note calculate the barching ratios and CP asymmetries of $B\to PP,PV$ decay modes in QCDF approach based on .

In [1]:
from dataclasses import dataclass
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import scipy as scp
import sympy as smp

In [2]:
from dataclasses import dataclass

@dataclass
class Quark:
    name: str  # 夸克名稱 (e.g., 'up', 'down', 'charm', 'strange', 'top', 'bottom')
    charge: float  # 電荷 (e.g., +2/3, -1/3)
    mass_GeVc2: float  # 質量 (單位: GeV/c^2)
    spin: float = 0.5  # 自旋 (默認為 0.5)

這個 `Quark` dataclass 包含以下欄位：

*   `name`: 夸克的名稱（例如 'up'、'down'）。
*   `charge`: 夸克的電荷，以分數形式表示（例如 `2/3` 或 `-1/3`）。
*   `mass_GeVc2`: 夸克的質量，單位為 GeV/c^2。
*   `spin`: 夸克的自旋，默認為 `0.5`。

現在您可以使用這個資料類別來創建夸克物件了。例如：

In [3]:
up_quark = Quark(name='up', charge=2/3, mass_GeVc2=0.002)
down_quark = Quark(name='down', charge=-1/3, mass_GeVc2=0.005)

print(up_quark)
print(down_quark)

Quark(name='up', charge=0.6666666666666666, mass_GeVc2=0.002, spin=0.5)
Quark(name='down', charge=-0.3333333333333333, mass_GeVc2=0.005, spin=0.5)


首先，讓我們建立一些 `Quark` 物件並將它們放入一個列表中：

In [4]:
charm_quark = Quark(name='charm', charge=2/3, mass_GeVc2=1.27)
strange_quark = Quark(name='strange', charge=-1/3, mass_GeVc2=0.095)
top_quark = Quark(name='top', charge=2/3, mass_GeVc2=173.2)
bottom_quark = Quark(name='bottom', charge=-1/3, mass_GeVc2=4.18)


quarks = [
    up_quark,
    down_quark,
    charm_quark,
    strange_quark,
    top_quark,
    bottom_quark
]

print("原始夸克列表：")
for quark in quarks:
    print(quark)

原始夸克列表：
Quark(name='up', charge=0.6666666666666666, mass_GeVc2=0.002, spin=0.5)
Quark(name='down', charge=-0.3333333333333333, mass_GeVc2=0.005, spin=0.5)
Quark(name='charm', charge=0.6666666666666666, mass_GeVc2=1.27, spin=0.5)
Quark(name='strange', charge=-0.3333333333333333, mass_GeVc2=0.095, spin=0.5)
Quark(name='top', charge=0.6666666666666666, mass_GeVc2=173.2, spin=0.5)
Quark(name='bottom', charge=-0.3333333333333333, mass_GeVc2=4.18, spin=0.5)


現在，我們可以依照 `mass_GeVc2` 屬性來對這個列表進行排序。您也可以根據 `name` 或 `charge` 等其他屬性來排序。

In [5]:
# 依照質量進行排序 (從小到大)
sorted_quarks_by_mass = sorted(quarks, key=lambda q: q.mass_GeVc2)

print("\n依質量排序後的夸克列表：")
for quark in sorted_quarks_by_mass:
    print(quark)


依質量排序後的夸克列表：
Quark(name='up', charge=0.6666666666666666, mass_GeVc2=0.002, spin=0.5)
Quark(name='down', charge=-0.3333333333333333, mass_GeVc2=0.005, spin=0.5)
Quark(name='strange', charge=-0.3333333333333333, mass_GeVc2=0.095, spin=0.5)
Quark(name='charm', charge=0.6666666666666666, mass_GeVc2=1.27, spin=0.5)
Quark(name='bottom', charge=-0.3333333333333333, mass_GeVc2=4.18, spin=0.5)
Quark(name='top', charge=0.6666666666666666, mass_GeVc2=173.2, spin=0.5)


如果您想依照夸克名稱進行排序 (字母順序)，可以使用 `key=lambda q: q.name`：

```python
sorted_quarks_by_name = sorted(quarks, key=lambda q: q.name)
print("\n依名稱排序後的夸克列表：")
for quark in sorted_quarks_by_name:
    print(quark)
```

In [6]:
sorted_quarks_by_name = sorted(quarks, key=lambda q: q.name)
print("\n依名稱排序後的夸克列表：")
for quark in sorted_quarks_by_name:
    print(quark)


依名稱排序後的夸克列表：
Quark(name='bottom', charge=-0.3333333333333333, mass_GeVc2=4.18, spin=0.5)
Quark(name='charm', charge=0.6666666666666666, mass_GeVc2=1.27, spin=0.5)
Quark(name='down', charge=-0.3333333333333333, mass_GeVc2=0.005, spin=0.5)
Quark(name='strange', charge=-0.3333333333333333, mass_GeVc2=0.095, spin=0.5)
Quark(name='top', charge=0.6666666666666666, mass_GeVc2=173.2, spin=0.5)
Quark(name='up', charge=0.6666666666666666, mass_GeVc2=0.002, spin=0.5)


現在，讓我們將排序後的夸克列表儲存為 CSV 檔案。我們可以使用 `pandas` 函式庫來完成這項任務。首先，我們需要將夸克物件列表轉換為 pandas DataFrame。

In [7]:
# Re-define Quark class and objects to ensure they are available
from dataclasses import dataclass
import pandas as pd

@dataclass
class Quark:
    name: str
    charge: float
    mass_GeVc2: float
    spin: float = 0.5

up_quark = Quark(name='up', charge=2/3, mass_GeVc2=0.002)
down_quark = Quark(name='down', charge=-1/3, mass_GeVc2=0.005)
charm_quark = Quark(name='charm', charge=2/3, mass_GeVc2=1.27)
strange_quark = Quark(name='strange', charge=-1/3, mass_GeVc2=0.095)
top_quark = Quark(name='top', charge=2/3, mass_GeVc2=173.2)
bottom_quark = Quark(name='bottom', charge=-1/3, mass_GeVc2=4.18)

quarks = [
    up_quark,
    down_quark,
    charm_quark,
    strange_quark,
    top_quark,
    bottom_quark
]

sorted_quarks_by_mass = sorted(quarks, key=lambda q: q.mass_GeVc2)

# 將排序後的夸克列表轉換為 DataFrame
quark_data = []
for quark in sorted_quarks_by_mass:
    quark_data.append({
        'name': quark.name,
        'charge': quark.charge,
        'mass_GeVc2': quark.mass_GeVc2,
        'spin': quark.spin
    })

quarks_df = pd.DataFrame(quark_data)

# 儲存為 CSV 檔案
csv_filename = 'quarks_sorted_by_mass.csv'
quarks_df.to_csv(csv_filename, index=False)

print(f"夸克數據已成功儲存至 {csv_filename}")

# 讀取 CSV 檔案以驗證
loaded_df = pd.read_csv(csv_filename)
print("\n已從 CSV 檔案讀取的數據：")
print(loaded_df.head())

夸克數據已成功儲存至 quarks_sorted_by_mass.csv

已從 CSV 檔案讀取的數據：
      name    charge  mass_GeVc2  spin
0       up  0.666667       0.002   0.5
1     down -0.333333       0.005   0.5
2  strange -0.333333       0.095   0.5
3    charm  0.666667       1.270   0.5
4   bottom -0.333333       4.180   0.5


In [8]:
# 計算不同電荷類型的數量
charge_counts = quarks_df['charge'].value_counts()

print("不同電荷類型的數量：")
print(charge_counts)

不同電荷類型的數量：
charge
 0.666667    3
-0.333333    3
Name: count, dtype: int64


現在，讓我們建立一個 `Meson` (介子) dataclass。介子是由一個夸克和一個反夸克組成的亞原子粒子。我們將使用之前定義的 `Quark` dataclass 作為其組成部分。

In [9]:
from dataclasses import dataclass

# 確保 Quark dataclass 在此處可用，以防獨立執行
@dataclass
class Quark:
    name: str  # 夸克名稱 (e.g., 'up', 'down', 'charm', 'strange', 'top', 'bottom')
    charge: float  # 電荷 (e.g., +2/3, -1/3)
    mass_GeVc2: float  # 質量 (單位: GeV/c^2)
    spin: float = 0.5  # 自旋 (默認為 0.5)

@dataclass
class Meson:
    name: str  # 介子名稱 (e.g., 'B0', 'pi+')
    quark1: Quark  # 第一個組成夸克
    quark2: Quark  # 第二個組成夸克 (反夸克)
    mass_GeVc2: float  # 介子的質量
    charge: float # 介子的電荷
    spin: float # 介子的自旋
    parity: str # 介子的宇稱


    @property
    def is_b_meson(self) -> bool:
        # B介子包含底夸克或反底夸克
        return 'bottom' in (self.quark1.name, self.quark2.name)

    @property
    def is_charm_less(self) -> bool:
        # 非魅輕介子不包含魅夸克或反魅夸克
        return 'charm' not in (self.quark1.name, self.quark2.name)

    @property
    def get_constituents(self):
        return f"({self.quark1.name}, {self.quark2.name} bar)"


現在我們可以創建一些夸克物件來組成介子。這裡我們會創建一些基礎夸克和它們的反夸克。

In [10]:
# 重新定義夸克以確保它們可用 (與之前的定義保持一致)
up_quark = Quark(name='up', charge=2/3, mass_GeVc2=0.002)
down_quark = Quark(name='down', charge=-1/3, mass_GeVc2=0.005)
charm_quark = Quark(name='charm', charge=2/3, mass_GeVc2=1.27)
strange_quark = Quark(name='strange', charge=-1/3, mass_GeVc2=0.095)
bottom_quark = Quark(name='bottom', charge=-1/3, mass_GeVc2=4.18)

# 反夸克通常具有相同的質量，但電荷相反。為簡潔起見，我們假設它們只是夸克名字加上 'bar'，電荷取反。
# 實際物理中反夸克是獨立的粒子，但這裡簡化處理。
anti_up_quark = Quark(name='up', charge=-2/3, mass_GeVc2=0.002)
anti_down_quark = Quark(name='down', charge=1/3, mass_GeVc2=0.005)
anti_charm_quark = Quark(name='charm', charge=-2/3, mass_GeVc2=1.27)
anti_strange_quark = Quark(name='strange', charge=1/3, mass_GeVc2=0.095)
anti_bottom_quark = Quark(name='bottom', charge=1/3, mass_GeVc2=4.18)


接下來，我們將創建一些 B 介子和非魅輕介子的實例：

In [13]:
# 創建 B 介子
B_zero = Meson(name='B0', quark1=down_quark, quark2=anti_bottom_quark, mass_GeVc2=5.279, charge=0, spin=0, parity='-')
B_plus = Meson(name='B+', quark1=up_quark, quark2=anti_bottom_quark, mass_GeVc2=5.279,  charge=1, spin=0, parity='-')
B_s_zero = Meson(name='Bs0', quark1=strange_quark, quark2=anti_bottom_quark, mass_GeVc2=5.367,  charge=0, spin=0, parity='-')

# 創建非魅輕介子 (不含魅夸克或底夸克)
# pi mesons
Pi_plus = Meson(name='pi+', quark1=up_quark, quark2=anti_down_quark, mass_GeVc2=0.139, charge=1, spin=0, parity='-')
Pi_zero = Meson(name='pi0', quark1=up_quark, quark2=anti_up_quark, mass_GeVc2=0.139, charge=0, spin=0, parity='-')# 實際為 (s anti-s + u anti-u)/sqrt(2)
Pi_minus = Meson(name='pi-', quark1=down_quark, quark2=anti_up_quark, mass_GeVc2=0.139, charge=-1, spin=0, parity='-')
#K mesons
K_plus = Meson(name='K+', quark1=up_quark, quark2=anti_strange_quark, mass_GeVc2=0.497, charge=1, spin=0, parity='-') # Corrected quark content
K_zero = Meson(name='K0', quark1=down_quark, quark2=anti_strange_quark, mass_GeVc2=0.497, charge=0, spin=0, parity='-') # Corrected quark content
Kb_zero = Meson(name='KBar0', quark1=strange_quark, quark2=anti_down_quark, mass_GeVc2=0.497, charge=0, spin=0, parity='-') # Corrected quark content
K_minus = Meson(name='K-', quark1=strange_quark, quark2=anti_up_quark, mass_GeVc2=0.497, charge=-1, spin=0, parity='-')
#vector mesons
Rho_plus = Meson(name='rho+', quark1=up_quark, quark2=anti_down_quark, mass_GeVc2=0.775, charge=1, spin=1, parity='-')
Rho_zero = Meson(name='rho0', quark1=up_quark, quark2=anti_up_quark, mass_GeVc2=0.775, charge=0, spin=1, parity='-')
Rho_minus = Meson(name='rho-', quark1=down_quark, quark2=anti_up_quark, mass_GeVc2=0.775, charge=-1, spin=1, parity='-') # 實際為 (u anti-u + d anti-d)/sqrt(2)
Kstar_plus = Meson(name='K*+', quark1=up_quark, quark2=anti_strange_quark, mass_GeVc2=0.497, charge=1, spin=1, parity='-') # Corrected quark content
Kstar_zero = Meson(name='K*0', quark1=down_quark, quark2=anti_strange_quark, mass_GeVc2=0.497, charge=0, spin=1, parity='-')
Kstar_minus = Meson(name='K*-', quark1=strange_quark, quark2=anti_up_quark, mass_GeVc2=0.497, charge=-1, spin=1, parity='-') # 實際為 (s anti-s + u anti-u)/sqrt(2)

# 顯示介子資訊
print("B 介子：")
print(f"  {B_zero.name} ({B_zero.get_constituents}): Mass = {B_zero.mass_GeVc2} GeV/c^2, Is B meson = {B_zero.is_b_meson}, Is charm-less = {B_zero.is_charm_less}")
print(f"  {B_plus.name} ({B_plus.get_constituents}): Mass = {B_plus.mass_GeVc2} GeV/c^2, Is B meson = {B_plus.is_b_meson}, Is charm-less = {B_plus.is_charm_less}")
print(f"  {B_s_zero.name} ({B_s_zero.get_constituents}): Mass = {B_s_zero.mass_GeVc2} GeV/c^2, Is B meson = {B_s_zero.is_b_meson}, Is charm-less = {B_s_zero.is_charm_less}")

print("\n非魅輕介子：")
print(f"  {Pi_plus.name} ({Pi_plus.get_constituents}): Mass = {Pi_plus.mass_GeVc2} GeV/c^2, Is B meson = {Pi_plus.is_b_meson}, Is charm-less = {Pi_plus.is_charm_less}")
print(f"  {K_zero.name} ({K_zero.get_constituents}): Mass = {K_zero.mass_GeVc2} GeV/c^2, Is B meson = {K_zero.is_b_meson}, Is charm-less = {K_zero.is_charm_less}")
print(f"  {Rho_zero.name} ({Rho_zero.get_constituents}): Mass = {Rho_zero.mass_GeVc2} GeV/c^2, Is B meson = {Rho_zero.is_b_meson}, Is charm-less = {Rho_zero.is_charm_less}")

B 介子：
  B0 ((down, bottom bar)): Mass = 5.279 GeV/c^2, Is B meson = True, Is charm-less = True
  B+ ((up, bottom bar)): Mass = 5.279 GeV/c^2, Is B meson = True, Is charm-less = True
  Bs0 ((strange, bottom bar)): Mass = 5.367 GeV/c^2, Is B meson = True, Is charm-less = True

非魅輕介子：
  pi+ ((up, down bar)): Mass = 0.139 GeV/c^2, Is B meson = False, Is charm-less = True
  K0 ((down, strange bar)): Mass = 0.497 GeV/c^2, Is B meson = False, Is charm-less = True
  rho0 ((up, up bar)): Mass = 0.775 GeV/c^2, Is B meson = False, Is charm-less = True


現在，讓我們定義一個函數，根據介子的夸克組成來自動判斷其類型。

In [ ]:
def get_meson_type(quark1: Quark, quark2: Quark) -> str:
    q_names = set([quark1.name, quark2.name]) # 使用集合方便檢查夸克是否存在

    if 'bottom' in q_names:
        return "B Meson"
    elif 'charm' in q_names:
        return "D Meson"
    elif 'strange' in q_names:
        return "K Meson"
    elif 'up' in q_names or 'down' in q_names:
        return "Pion-like Meson" # 由上夸克或下夸克組成
    else:
        return "Unknown Meson Type"


讓我們使用這個函數來判斷之前創建的介子的類型：

In [15]:
def get_meson_type(quark1: Quark, quark2: Quark) -> str:
    q_names = set([quark1.name, quark2.name]) # 使用集合方便檢查夸克是否存在

    if 'bottom' in q_names:
        return "B Meson"
    elif 'charm' in q_names:
        return "D Meson"
    elif 'strange' in q_names:
        return "K Meson"
    elif 'up' in q_names or 'down' in q_names:
        return "Pion-like Meson" # 由上夸克或下夸克組成
    else:
        return "Unknown Meson Type"

print("判斷介子類型：")
print(f"  {B_zero.name}: {get_meson_type(B_zero.quark1, B_zero.quark2)}")
print(f"  {B_plus.name}: {get_meson_type(B_plus.quark1, B_plus.quark2)}")
print(f"  {B_s_zero.name}: {get_meson_type(B_s_zero.quark1, B_s_zero.quark2)}")
print(f"  {Pi_plus.name}: {get_meson_type(Pi_plus.quark1, Pi_plus.quark2)}")
print(f"  {K_zero.name}: {get_meson_type(K_zero.quark1, K_zero.quark2)}")
print(f"  {Rho_zero.name}: {get_meson_type(Rho_zero.quark1, Rho_zero.quark2)}")

判斷介子類型：
  B0: B Meson
  B+: B Meson
  Bs0: B Meson
  pi+: Pion-like Meson
  K0: K Meson
  rho0: Pion-like Meson
